# Sensor synchronization

Study stale measurements and future-data leakage before integrating asynchronous sensors.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Timestamped streams

Each buffer accepts strictly increasing timestamps and evicts its oldest item at capacity.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.experiments.streams import ObservationBuffer, TimedObservation, synchronize

camera = ObservationBuffer(capacity=100)
lidar = ObservationBuffer(capacity=100)
for index in range(30):
    camera.append(TimedObservation(index / 10, f"camera-{index}"))
for index in range(15):
    lidar.append(TimedObservation(index / 5 + 0.03, f"lidar-{index}"))

## Query only past measurements

The nearest future observation must never be used to make a current decision.

In [ ]:
query = 1.05
result = synchronize({"camera": camera, "lidar": lidar}, query, maximum_age=0.3)
print(result)
assert result["ready"]
assert camera.latest_before(query).time <= query
assert lidar.latest_before(query).time <= query

## Age over time

Readiness is a policy about data freshness, not merely whether a buffer is nonempty.

In [ ]:
times = np.linspace(0, 3.5, 120)
ages = {"camera": [], "lidar": []}
ready = []
for time in times:
    sample = synchronize({"camera": camera, "lidar": lidar}, float(time), 0.25)
    ready.append(sample["ready"])
    for name in ages:
        age = sample["ages"][name]
        ages[name].append(np.nan if age is None else age)
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for name, values in ages.items():
    axes[0].plot(times, values, label=name)
axes[0].axhline(0.25, linestyle="--", color="black")
axes[0].set_ylabel("Age (s)")
axes[0].legend()
axes[1].plot(times, ready)
axes[1].set_ylabel("Ready")
axes[1].set_xlabel("Query time (s)")
plt.show()

## Stale sensor

A missing recent sample yields None rather than silently reusing stale state.

In [ ]:
late = synchronize({"camera": camera, "lidar": lidar}, 4.0, maximum_age=0.25)
assert not late["ready"]
assert late["values"] == {"camera": None, "lidar": None}
print(late)
interval = camera.between(1.0, 1.3)
print("Interval observations:", interval)

## Ordering failure

Duplicate timestamps need an explicit upstream policy. This buffer rejects them.

In [ ]:
try:
    camera.append(TimedObservation(1.0, "out-of-order"))
except ValueError as error:
    print("Expected rejection:", error)
else:
    raise AssertionError("Out-of-order observation was accepted")
print("Buffer lengths:", len(camera), len(lidar))

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.